# Ames Housing Price Prediction

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

# Data Understanding

In [ ]:
df = pd.read_csv("dataset/Ames_Housing_Dataset.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['SalePrice'].describe()

# Data Cleaning

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True,inplace=True)

In [ ]:
df.drop(['Id','Utilities'],axis=1,inplace=True)

In [ ]:
none_cols = ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']
df[none_cols] = df[none_cols].fillna('None')

In [ ]:
df.drop('GarageYrBlt',axis=1,inplace=True)

In [ ]:
df['MSSubClass'] = df['MSSubClass'].astype(str)

In [ ]:
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
df['TotalBath'] = df['FullBath'] + 0.5*df['HalfBath'] + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath']

In [ ]:
df.shape

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)

# Train Test Split

In [ ]:
X = df.drop('SalePrice',axis = 1)
y = df['SalePrice']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,shuffle=True,random_state=42)

In [ ]:
X_train.reset_index(drop=True,inplace=True)
X_test.reset_index(drop=True,inplace=True)

y_train.reset_index(drop=True,inplace=True)
y_test.reset_index(drop=True,inplace=True)

In [ ]:
y_test_orig = y_test.copy()
y_train = np.log1p(y_train)
y_test = np.log1p(y_test)

In [ ]:
y_train.skew()

# Missing Values

In [ ]:
X_train.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
X_train[['LotFrontage', 'MasVnrArea']].skew()

In [ ]:
median = ['LotFrontage', 'MasVnrArea']
impute = SimpleImputer(strategy='median')
for col in median:
    X_train[col] = impute.fit_transform(X_train[[col]])
    X_test[col] = impute.transform(X_test[[col]])

In [ ]:
mode = ['Electrical']
impute = SimpleImputer(strategy='most_frequent')
for col in mode:
    X_train[[col]] = impute.fit_transform(X_train[[col]])
    X_test[[col]] = impute.transform(X_test[[col]])

In [ ]:
X_train.isnull().sum().sum() , X_test.isnull().sum().sum()

In [ ]:
X_train.duplicated().sum() , X_test.duplicated().sum()

# Data Visualization (EDA)

In [ ]:
df_train = pd.concat([X_train,y_train],axis=1)

In [ ]:
plt.hist(df_train['SalePrice'],bins=40)
plt.show()

In [ ]:
df_train['SalePrice'].skew()

In [ ]:
top_cols = df_train.corr(numeric_only=True)['SalePrice'].abs().sort_values(ascending=False).head(12).index
plt.figure(figsize=(12,8))
sns.heatmap(df_train[top_cols].corr(),annot=True,cmap='Blues',fmt='0.2f')
plt.show()

In [ ]:
num_cols = ['OverallQual', 'TotalSF', 'GrLivArea', 'GarageCars', 'TotalBath', 'GarageArea']
plt.figure(figsize=(18,8))
for n , col in enumerate(num_cols):
    plt.subplot(2,3,n+1)
    plt.hist(df_train[col],bins=30)
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(18,8))
for n , col in enumerate(num_cols):
    plt.subplot(2,3,n+1)
    sns.boxplot(data=df_train,x=col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(18,8))
for n , col in enumerate(num_cols):
    plt.subplot(2,3,n+1)
    sns.scatterplot(data=df_train,x=col,y='SalePrice',alpha=0.4)
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(data=df_train,x='Condition1',y='SalePrice',showfliers=False)
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(data=df_train,x='SaleType',y='SalePrice',showfliers=False)
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(data=df_train,x='Condition2',y='SalePrice',showfliers=False)
plt.xticks(rotation=45)
plt.show()

# Data Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder , OrdinalEncoder , OneHotEncoder , StandardScaler

In [ ]:
X_train.select_dtypes(exclude='number').nunique()

In [ ]:
binary_cols = ['Street', 'CentralAir']
lb = LabelEncoder()
for col in binary_cols:
    X_train[col] = lb.fit_transform(X_train[col])
    X_test[col] = lb.transform(X_test[col])

In [ ]:
ordinal_cols = {"ExterQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"], "ExterCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"], "BsmtQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"], "BsmtCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"], "HeatingQC": ["None", "Po", "Fa", "TA", "Gd", "Ex"], "KitchenQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"], "FireplaceQu": ["None", "Po", "Fa", "TA", "Gd", "Ex"], "GarageQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"], "GarageCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"], "PoolQC": ["None", "Po", "Fa", "TA", "Gd", "Ex"], "BsmtExposure": ["None", "No", "Mn", "Av", "Gd"], "BsmtFinType1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"], "BsmtFinType2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"], "GarageFinish": ["None", "Unf", "RFn", "Fin"], "LandSlope": ["Gtl", "Mod", "Sev"], "LotShape": ["IR3", "IR2", "IR1", "Reg"], "PavedDrive": ["N", "P", "Y"]}

In [ ]:
for col , order in ordinal_cols.items():
    oe = OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1)
    X_train[[col]] = oe.fit_transform(X_train[[col]])
    X_test[[col]] = oe.transform(X_test[[col]])

In [ ]:
cat_cols = ['MSSubClass', 'MSZoning', 'Alley', 'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'Electrical', 'Functional', 'GarageType', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']

In [ ]:
onehot = OneHotEncoder(handle_unknown='ignore')
onehot.fit(X_train[cat_cols])

In [ ]:
temp = onehot.transform(X_train[cat_cols]).toarray()
temp = pd.DataFrame(temp,columns=onehot.get_feature_names_out(cat_cols))
X_train = pd.concat([X_train,temp],axis=1)
X_train.drop(cat_cols,axis=1,inplace=True)

In [ ]:
temp = onehot.transform(X_test[cat_cols]).toarray()
temp = pd.DataFrame(temp,columns=onehot.get_feature_names_out(cat_cols))
X_test = pd.concat([X_test,temp],axis=1)
X_test.drop(cat_cols,axis=1,inplace=True)

In [ ]:
X_train.shape , X_test.shape

# Feature Scaling

In [ ]:
scale_cols = ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'TotalSF', 'HouseAge', 'RemodAge', 'TotalBath']
scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

In [ ]:
X_train.head()

In [ ]:
X_train.isnull().sum().sum() , X_test.isnull().sum().sum()

In [ ]:
list(X_train.columns) == list(X_test.columns)

# Build Machine Learning Models

In [ ]:
from tqdm import tqdm
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
models = {
    "Linear Regression":LinearRegression(),
    "Ridge":Ridge(alpha=1),
    "Lasso":Lasso(alpha=0.001,max_iter=10000),
    "ElasticNet":ElasticNet(alpha=0.001,l1_ratio=0.5,max_iter=10000),
    "KNN":KNeighborsRegressor(n_neighbors=5),
    "Decision Tree":DecisionTreeRegressor(max_depth=10,random_state=42),
    "Random Forest":RandomForestRegressor(n_estimators=100,max_depth=10,random_state=42),
    "ExtraTrees":ExtraTreesRegressor(n_estimators=100,random_state=42),
    "SVR":SVR(kernel='rbf',C=100),
    "Gradient Boosting":GradientBoostingRegressor(n_estimators=100,learning_rate=0.1,random_state=42),
    "AdaBoost":AdaBoostRegressor(n_estimators=50,random_state=42),
    "XGBoost":XGBRegressor(n_estimators=100,learning_rate=0.1,random_state=42,verbosity=0),
    "LightGBM":LGBMRegressor(n_estimators=100,learning_rate=0.1,random_state=42,verbose=-1)
}

In [ ]:
def get_fit_data(model_name):
    if model_name in ['SVM','KNN'] and len(X_train) > 20000:
        return X_train[:20000] , y_train[:20000]
    return X_train , y_train

In [ ]:
def regression_metrics(model, X, y_actual):
    y_pred = model.predict(X)
    MAE = mean_absolute_error(y_actual, y_pred)
    MSE = mean_squared_error(y_actual, y_pred)
    RMSE = np.sqrt(MSE)
    R2 = r2_score(y_actual, y_pred)
    return MAE, MSE, RMSE, R2

In [ ]:
errors={}
for model_name , model in tqdm(models.items()):
    X_fit , y_fit = get_fit_data(model_name)
    model.fit(X_fit,y_fit)
    MAE_Train, MSE_Train, RMSE_Train, R2_Train = regression_metrics(model, X_fit, y_fit)
    MAE_Test, MSE_Test, RMSE_Test, R2_Test = regression_metrics(model, X_test, y_test)

    errors[model_name] = {
        "MAE Train":MAE_Train,
        "MAE Test":MAE_Test,
        "MSE Train":MSE_Train,
        "MSE Test":MSE_Test,
        "RMSE Train":RMSE_Train,
        "RMSE Test":RMSE_Test,
        "R2 Train":R2_Train,
        "R2 Test":R2_Test
    }

# Models Comparison (Before Tuning)

In [ ]:
errors_df = pd.DataFrame(errors).T
errors_df

In [ ]:
cols = [
    ['MAE Train', 'MAE Test'],
    ['MSE Train', 'MSE Test'],
    ['RMSE Train', 'RMSE Test'],
    ['R2 Train', 'R2 Test']
]

plt.figure(figsize=(20,20))
for i, col in enumerate(cols):
    ax = plt.subplot(2, 2, i+1)
    errors_df[col].plot.bar(ax=ax)
    ax.set_title(col[0].split(" ")[0] + ' Comparison')
plt.tight_layout()
plt.show()

# Cross-Validation

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
cv_results = {}
for model_name , model in tqdm(models.items()):
    X_cv , y_cv = get_fit_data(model_name)
    scores = cross_validate(model, X_cv, y_cv, cv=cv, scoring=['r2','neg_root_mean_squared_error'], n_jobs=-1)
    cv_results[model_name] = {
        "CV R2 mean":scores['test_r2'].mean(),
        "CV R2 std":scores['test_r2'].std(),
        "CV RMSE mean":-scores['test_neg_root_mean_squared_error'].mean(),
        "CV RMSE std":scores['test_neg_root_mean_squared_error'].std()
    }

In [ ]:
cv_df = pd.DataFrame(cv_results).T.sort_values(by='CV R2 mean',ascending=False)
cv_df

In [ ]:
cv_df[['CV R2 mean']].plot.bar(figsize=(14,6))
plt.show()

# GridSearchCV

In [ ]:
param_grids = {
    "Linear Regression": {"fit_intercept": [True, False]},
    "Ridge": {"alpha": [0.01, 0.1, 1, 10, 100]},
    "Lasso": {"alpha": [0.0001, 0.001, 0.01, 0.1]},
    "ElasticNet": {"alpha": [0.001, 0.01, 0.1], "l1_ratio": [0.2, 0.5, 0.8]},
    "KNN": {"n_neighbors": [3, 5, 11, 21], "weights": ["uniform", "distance"]},
    "Decision Tree": {"max_depth": [3, 5, 10, 15], "min_samples_leaf": [1, 5, 20]},
    "Random Forest": {"n_estimators": [100, 200], "max_depth": [10, 20], "min_samples_leaf": [1, 5]},
    "ExtraTrees": {"n_estimators": [100, 200], "max_depth": [10, None]},
    "SVR": {"C": [1, 10, 100], "epsilon": [0.01, 0.1]},
    "Gradient Boosting": {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [3, 5]},
    "AdaBoost": {"n_estimators": [50, 100, 200], "learning_rate": [0.1, 1]},
    "XGBoost": {"n_estimators": [100, 300], "learning_rate": [0.05, 0.1], "max_depth": [3, 6]},
    "LightGBM": {"n_estimators": [100, 300], "learning_rate": [0.05, 0.1], "num_leaves": [15, 31]}
}

In [ ]:
candidates = [name for name in cv_df.index if name in param_grids][:3]
candidates

In [ ]:
grids = {}
grid_rows = {}

for name in candidates:
    X_g , y_g = get_fit_data(name)
    grid = GridSearchCV(estimator=models[name], param_grid=param_grids[name], cv=cv, scoring='r2', n_jobs=-1)
    grid.fit(X_g, y_g)
    grids[name] = grid

    print(name)
    print("best_params_ :", grid.best_params_)
    print("best_score_  :", round(grid.best_score_,4))
    print()

    grid_rows[name] = {
        "CV R2 before":cv_df.loc[name,"CV R2 mean"],
        "CV R2 after (best_score_)":grid.best_score_,
        "best_params":grid.best_params_
    }

In [ ]:
grid_df = pd.DataFrame(grid_rows).T
grid_df["Improvement"] = grid_df["CV R2 after (best_score_)"] - grid_df["CV R2 before"]
grid_df

In [ ]:
best_name = grid_df["CV R2 after (best_score_)"].astype(float).idxmax()
best_params = grids[best_name].best_params_
best_tuned = grids[best_name].best_estimator_
best_default = models[best_name]
best_name , best_params

# Final Evaluation on Test Set

In [ ]:
rows = {}
for label , m in [("Before tuning",best_default),("After tuning",best_tuned)]:
    MAE, MSE, RMSE, R2 = regression_metrics(m, X_test, y_test)
    rows[label] = {"MAE":MAE,"MSE":MSE,"RMSE":RMSE,"R2":R2}

final_df = pd.DataFrame(rows).T
final_df

In [ ]:
rows = {}
for label , m in [("Before tuning",best_default),("After tuning",best_tuned)]:
    pred = np.expm1(m.predict(X_test))
    MSE = mean_squared_error(y_test_orig, pred)
    rows[label] = {"MAE":mean_absolute_error(y_test_orig, pred),"MSE":MSE,"RMSE":np.sqrt(MSE),"R2":r2_score(y_test_orig, pred)}

final_orig_df = pd.DataFrame(rows).T
final_orig_df

In [ ]:
y_true_plot = y_test_orig
y_pred_plot = np.expm1(best_tuned.predict(X_test))

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y_true_plot,y_pred_plot,alpha=0.4)
lims = [min(y_true_plot.min(),y_pred_plot.min()),max(y_true_plot.max(),y_pred_plot.max())]
plt.plot(lims,lims,"r--")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted - " + best_name)
plt.show()

In [ ]:
residuals = y_true_plot - y_pred_plot
fig , ax = plt.subplots(1,2,figsize=(14,5))
ax[0].scatter(y_pred_plot,residuals,alpha=0.4)
ax[0].axhline(0,color="red",linestyle="--")
ax[0].set_xlabel("Predicted")
ax[0].set_ylabel("Residual")
ax[0].set_title("Residuals vs Predicted")
ax[1].hist(residuals,bins=40)
ax[1].set_title("Residuals distribution")
plt.show()

In [ ]:
print("Model:", best_name, best_params)
print("Mean absolute error:", round(np.abs(residuals).mean(),3), "| mean target:", round(y_true_plot.mean(),3))
print("Mean absolute error as share of mean target:", round(np.abs(residuals).mean()/y_true_plot.mean()*100,1), "%")
print("Mean residual (bias):", round(residuals.mean(),3))

# Feature Importance

In [ ]:
if hasattr(best_tuned, "feature_importances_"):
    imp = pd.Series(best_tuned.feature_importances_,index=X_train.columns)
elif hasattr(best_tuned, "coef_"):
    imp = pd.Series(np.abs(np.ravel(best_tuned.coef_)),index=X_train.columns)
else:
    imp = pd.Series(models["Random Forest"].feature_importances_,index=X_train.columns)
imp = imp.sort_values(ascending=False).head(15)
sns.barplot(x=imp.values,y=imp.index,orient="h")
plt.xlabel("Importance")
plt.title("Feature Importance - " + best_name)
plt.show()